# Spark 学习笔记 - NYC 出租车数据分析

使用真实的纽约出租车数据学习 PySpark。

**数据说明：**
- PostgreSQL: 100,000 行 (用于学习 JDBC 连接)
- Parquet 文件: 3,066,766 行 (用于大数据处理练习)

## 1. 创建 SparkSession

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("NYC Taxi Analysis") \
    .master("local[*]") \
    .config("spark.jars", "/opt/spark/jars/postgresql-42.6.0.jar") \
    .config("spark.driver.extraClassPath", "/opt/spark/jars/postgresql-42.6.0.jar") \
    .getOrCreate()

print(f"Spark version: {spark.version}")
print("SparkSession 创建成功!")

## 2. 从 PostgreSQL 读取数据

In [ ]:
# 数据库连接配置
jdbc_url = "jdbc:postgresql://postgres:5432/dataplatform"
db_properties = {
    "user": "dataplatform",
    "password": "devpassword",
    "driver": "org.postgresql.Driver"
}

# 读取数据
df = spark.read.jdbc(jdbc_url, "nyc_taxi_trips", properties=db_properties)

print(f"读取到 {df.count():,} 行数据")
df.printSchema()

In [ ]:
# 查看前几行
df.show(5)

## 3. 基础查询操作

In [ ]:
# 选择特定列
df.select("tpep_pickup_datetime", "passenger_count", "trip_distance", "total_amount").show(5)

In [ ]:
# 过滤: 找出车费超过 50 美元的行程
df.filter(df.total_amount > 50).select(
    "tpep_pickup_datetime", "trip_distance", "total_amount"
).show(10)

In [ ]:
# 排序: 按行程距离降序
df.orderBy(df.trip_distance.desc()).select(
    "tpep_pickup_datetime", "trip_distance", "total_amount"
).show(10)

## 4. 聚合分析

In [ ]:
from pyspark.sql.functions import count, avg, sum, max, min, round

# 整体统计
df.agg(
    count("*").alias("总行程数"),
    round(avg("trip_distance"), 2).alias("平均距离(英里)"),
    round(avg("total_amount"), 2).alias("平均费用($)"),
    round(sum("total_amount"), 2).alias("总收入($)")
).show()

In [ ]:
# 按乘客数量分组统计
df.groupBy("passenger_count").agg(
    count("*").alias("行程数"),
    round(avg("trip_distance"), 2).alias("平均距离"),
    round(avg("total_amount"), 2).alias("平均费用")
).orderBy("passenger_count").show()

In [ ]:
# 按支付方式分组 (1=信用卡, 2=现金)
df.groupBy("payment_type").agg(
    count("*").alias("行程数"),
    round(avg("tip_amount"), 2).alias("平均小费")
).orderBy("payment_type").show()

## 5. 使用 SQL 查询

In [ ]:
# 注册临时视图
df.createOrReplaceTempView("taxi_trips")

In [ ]:
# SQL: 每小时的行程数量
spark.sql("""
    SELECT 
        HOUR(tpep_pickup_datetime) as hour,
        COUNT(*) as trip_count,
        ROUND(AVG(total_amount), 2) as avg_fare
    FROM taxi_trips
    GROUP BY HOUR(tpep_pickup_datetime)
    ORDER BY hour
""").show(24)

In [ ]:
# SQL: 找出最赚钱的上车地点 Top 10
spark.sql("""
    SELECT 
        PULocationID as pickup_location,
        COUNT(*) as trip_count,
        ROUND(SUM(total_amount), 2) as total_revenue,
        ROUND(AVG(total_amount), 2) as avg_fare
    FROM taxi_trips
    GROUP BY PULocationID
    ORDER BY total_revenue DESC
    LIMIT 10
""").show()

## 6. SQL vs PySpark DataFrame API 对比

两种方式实现相同的功能，选择你更熟悉的方式。

| 特点 | SQL | PySpark DataFrame |
|------|-----|-------------------|
| **学习曲线** | 低 (熟悉 SQL 即可) | 中等 (需学习 API) |
| **类型安全** | 运行时检查 | 编译时检查 |
| **IDE 支持** | 字符串，无补全 | 有代码补全 |
| **复杂逻辑** | 复杂查询难维护 | 可拆分、复用 |
| **调试** | 较难 | 可逐步调试 |

**建议：** 简单查询用 SQL，复杂数据处理用 DataFrame API。

### 6.1 基础查询对比: SELECT + WHERE

In [ ]:
import time

# SQL 方式: 查找车费超过 30 美元的短途行程
start_time = time.time()

sql_result = spark.sql("""
    SELECT tpep_pickup_datetime, trip_distance, fare_amount, tip_amount, total_amount
    FROM taxi_trips
    WHERE total_amount > 30 AND trip_distance < 2
    ORDER BY total_amount DESC
    LIMIT 5
""")
sql_result.show()

sql_time = time.time() - start_time
print(f"=== SQL 方式耗时: {sql_time:.3f} 秒 ===")

In [ ]:
# PySpark DataFrame 方式: 同样的查询
from pyspark.sql.functions import col

start_time = time.time()

pyspark_result = df \
    .select("tpep_pickup_datetime", "trip_distance", "fare_amount", "tip_amount", "total_amount") \
    .filter((col("total_amount") > 30) & (col("trip_distance") < 2)) \
    .orderBy(col("total_amount").desc()) \
    .limit(5)
pyspark_result.show()

pyspark_time = time.time() - start_time
print(f"=== PySpark DataFrame 方式耗时: {pyspark_time:.3f} 秒 ===")
print(f"\n时间对比: SQL={sql_time:.3f}s vs PySpark={pyspark_time:.3f}s")

### 6.2 聚合对比: GROUP BY

In [ ]:
# SQL 方式: 按乘客数统计
start_time = time.time()

spark.sql("""
    SELECT 
        passenger_count,
        COUNT(*) as trips,
        ROUND(AVG(total_amount), 2) as avg_fare,
        ROUND(AVG(tip_amount), 2) as avg_tip
    FROM taxi_trips
    WHERE passenger_count IS NOT NULL AND passenger_count > 0
    GROUP BY passenger_count
    ORDER BY passenger_count
""").show()

sql_time_groupby = time.time() - start_time
print(f"=== SQL 方式耗时: {sql_time_groupby:.3f} 秒 ===")

In [ ]:
# PySpark DataFrame 方式: 同样的聚合
from pyspark.sql.functions import count, avg, round

start_time = time.time()

df \
    .filter((col("passenger_count").isNotNull()) & (col("passenger_count") > 0)) \
    .groupBy("passenger_count") \
    .agg(
        count("*").alias("trips"),
        round(avg("total_amount"), 2).alias("avg_fare"),
        round(avg("tip_amount"), 2).alias("avg_tip")
    ) \
    .orderBy("passenger_count") \
    .show()

pyspark_time_groupby = time.time() - start_time
print(f"=== PySpark DataFrame 方式耗时: {pyspark_time_groupby:.3f} 秒 ===")
print(f"\n时间对比: SQL={sql_time_groupby:.3f}s vs PySpark={pyspark_time_groupby:.3f}s")

### 6.3 条件逻辑对比: CASE WHEN vs when().otherwise()

In [ ]:
# SQL 方式: 使用 CASE WHEN 分类行程
start_time = time.time()

spark.sql("""
    SELECT 
        CASE 
            WHEN trip_distance < 1 THEN '短途 (<1英里)'
            WHEN trip_distance < 5 THEN '中途 (1-5英里)'
            WHEN trip_distance < 10 THEN '长途 (5-10英里)'
            ELSE '超长途 (>10英里)'
        END as trip_category,
        COUNT(*) as trip_count,
        ROUND(AVG(total_amount), 2) as avg_fare
    FROM taxi_trips
    GROUP BY 
        CASE 
            WHEN trip_distance < 1 THEN '短途 (<1英里)'
            WHEN trip_distance < 5 THEN '中途 (1-5英里)'
            WHEN trip_distance < 10 THEN '长途 (5-10英里)'
            ELSE '超长途 (>10英里)'
        END
    ORDER BY trip_count DESC
""").show()

sql_time_case = time.time() - start_time
print(f"=== SQL 方式耗时: {sql_time_case:.3f} 秒 ===")

In [ ]:
# PySpark DataFrame 方式: 使用 when().otherwise()
from pyspark.sql.functions import when

start_time = time.time()

# 定义分类逻辑 (可复用!)
trip_category = when(col("trip_distance") < 1, "短途 (<1英里)") \
    .when(col("trip_distance") < 5, "中途 (1-5英里)") \
    .when(col("trip_distance") < 10, "长途 (5-10英里)") \
    .otherwise("超长途 (>10英里)")

df \
    .withColumn("trip_category", trip_category) \
    .groupBy("trip_category") \
    .agg(
        count("*").alias("trip_count"),
        round(avg("total_amount"), 2).alias("avg_fare")
    ) \
    .orderBy(col("trip_count").desc()) \
    .show()

pyspark_time_case = time.time() - start_time
print(f"=== PySpark DataFrame 方式耗时: {pyspark_time_case:.3f} 秒 ===")
print(f"\n时间对比: SQL={sql_time_case:.3f}s vs PySpark={pyspark_time_case:.3f}s")

### 6.4 窗口函数对比: OVER() vs Window

In [ ]:
# SQL 方式: 计算每个支付方式中费用排名
start_time = time.time()

spark.sql("""
    SELECT 
        payment_type,
        total_amount,
        RANK() OVER (PARTITION BY payment_type ORDER BY total_amount DESC) as rank
    FROM taxi_trips
    WHERE payment_type IN (1, 2)
""").filter("rank <= 3").orderBy("payment_type", "rank").show(10)

sql_time_window = time.time() - start_time
print(f"=== SQL 方式耗时: {sql_time_window:.3f} 秒 ===")

In [ ]:
# PySpark DataFrame 方式: 使用 Window 函数
from pyspark.sql.window import Window
from pyspark.sql.functions import rank

start_time = time.time()

# 定义窗口规范
window_spec = Window.partitionBy("payment_type").orderBy(col("total_amount").desc())

df \
    .filter(col("payment_type").isin([1, 2])) \
    .withColumn("rank", rank().over(window_spec)) \
    .filter(col("rank") <= 3) \
    .select("payment_type", "total_amount", "rank") \
    .orderBy("payment_type", "rank") \
    .show(10)

pyspark_time_window = time.time() - start_time
print(f"=== PySpark DataFrame 方式耗时: {pyspark_time_window:.3f} 秒 ===")
print(f"\n时间对比: SQL={sql_time_window:.3f}s vs PySpark={pyspark_time_window:.3f}s")

### 6.5 语法速查表

| 操作 | SQL | PySpark DataFrame |
|------|-----|-------------------|
| 选择列 | `SELECT col1, col2` | `.select("col1", "col2")` |
| 过滤 | `WHERE condition` | `.filter(condition)` |
| 排序 | `ORDER BY col DESC` | `.orderBy(col("x").desc())` |
| 限制行数 | `LIMIT 10` | `.limit(10)` |
| 分组 | `GROUP BY col` | `.groupBy("col")` |
| 聚合 | `COUNT(*), AVG(col)` | `.agg(count("*"), avg("col"))` |
| 别名 | `AS alias` | `.alias("alias")` |
| 条件 | `CASE WHEN...END` | `when().otherwise()` |
| 新增列 | `SELECT *, expr AS new` | `.withColumn("new", expr)` |
| 去重 | `DISTINCT` | `.distinct()` |
| 连接 | `JOIN ON` | `.join(df2, condition)` |
| 窗口 | `OVER(PARTITION BY...)` | `Window.partitionBy(...)` |

**记住：** 两种方式最终会生成相同的执行计划，性能没有区别！

### 6.6 性能对比总结

In [ ]:
### 6.7 语法速查表

| 操作 | SQL | PySpark DataFrame |
|------|-----|-------------------|
| 选择列 | `SELECT col1, col2` | `.select("col1", "col2")` |
| 过滤 | `WHERE condition` | `.filter(condition)` |
| 排序 | `ORDER BY col DESC` | `.orderBy(col("x").desc())` |
| 限制行数 | `LIMIT 10` | `.limit(10)` |
| 分组 | `GROUP BY col` | `.groupBy("col")` |
| 聚合 | `COUNT(*), AVG(col)` | `.agg(count("*"), avg("col"))` |
| 别名 | `AS alias` | `.alias("alias")` |
| 条件 | `CASE WHEN...END` | `when().otherwise()` |
| 新增列 | `SELECT *, expr AS new` | `.withColumn("new", expr)` |
| 去重 | `DISTINCT` | `.distinct()` |
| 连接 | `JOIN ON` | `.join(df2, condition)` |
| 窗口 | `OVER(PARTITION BY...)` | `Window.partitionBy(...)` |

**重要提示：** 
- SQL 和 PySpark DataFrame API 生成**相同的执行计划**
- 性能上没有本质区别，选择你更熟悉的方式即可
- 简单查询推荐 SQL (更直观)，复杂逻辑推荐 DataFrame API (更灵活、可复用)

## 6. 读取 Parquet 文件 (大数据集)

Parquet 是列式存储格式，Spark 读取更快，适合大数据处理。

In [ ]:
# 读取完整数据集 (300万行)
df_full = spark.read.parquet("/workspace/data/nyc_taxi_2023_01.parquet")
print(f"Parquet 数据集: {df_full.count():,} 行")

In [ ]:
# 对比: 用大数据集做同样的分析
df_full.createOrReplaceTempView("taxi_trips_full")

spark.sql("""
    SELECT 
        DATE(tpep_pickup_datetime) as date,
        COUNT(*) as trip_count,
        ROUND(SUM(total_amount), 2) as daily_revenue
    FROM taxi_trips_full
    GROUP BY DATE(tpep_pickup_datetime)
    ORDER BY date
    LIMIT 10
""").show()

## 7. 数据清洗练习

In [ ]:
from pyspark.sql.functions import col, when

# 检查空值
df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns]).show()

In [ ]:
# 过滤异常数据: 距离>0, 费用>0, 乘客数>0
df_clean = df.filter(
    (col("trip_distance") > 0) & 
    (col("total_amount") > 0) & 
    (col("passenger_count") > 0)
)

print(f"原始数据: {df.count():,} 行")
print(f"清洗后: {df_clean.count():,} 行")
print(f"移除了: {df.count() - df_clean.count():,} 行异常数据")

## 8. 练习题

试着完成以下分析：

1. 找出小费最高的 10 笔行程
2. 计算每种支付方式的小费率 (tip_amount / fare_amount)
3. 找出哪个时段 (早高峰/晚高峰/深夜) 行程最多
4. 分析行程距离和费用的关系

In [ ]:
# 练习 1: 小费最高的 10 笔行程


In [ ]:
# 练习 2: 每种支付方式的小费率


In [ ]:
# 练习 3: 时段分析


In [ ]:
# 练习 4: 距离与费用关系


## 9. 关闭 SparkSession

In [ ]:
# 使用完毕后关闭 (释放资源)
# spark.stop()